# Vector-Valued GP Adaptive Sampling

We wish to adaptively sample a GP like $f \sim \text{GP}(\mu, k)$ such that $f: \mathbb{R}^n \rightarrow \mathbb{R}^m$ where $\mu$ and $k$ are the prior mean and covariance kernel, respectively.

Similarly to the case in which $f: \mathbb{R} \rightarrow \mathbb{R}$, we (iteratively) update the posterior with the point in our domain associated with highest variance.

In [ ]:
import matplotlib.axes as mpl_axes
import matplotlib.cm as mpl_cm
import matplotlib.colors as mpl_colors
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.insert(0, '../')

import UncertainSCI.gp as gp


MESH_KWARGS = dict(
    truth = dict(
        vmin=-3.,
        vmax=3.,
        cmap='Spectral_r'
    ),
    mean = dict(
        vmin=-3.,
        vmax=3.,
        cmap='Spectral_r'
    ),
    variance = dict(
        vmin=0.,
        vmax=3.,
        cmap='RdYlGn_r'
    ),
    error = dict(vmin=0.,
        vmax=3.,
        cmap='RdYlGn_r')
)
FIGSIZE = (6.4, 4.8)
PLOT_EVERY = 10

N_SMP = int(1e5)

E1_PTS = 40
E1LIM = (0, 2)
E2_PTS = 20
E2LIM = (0, 1)

CMAP = mpl_cm.get_cmap('cool')
CMAP.set_bad('#000000')
CMAP.set_over('#ff0000')
NORM = mpl_colors.Normalize(vmin=0., vmax=3.)
M = mpl_cm.ScalarMappable(norm=NORM, cmap=CMAP)


In [ ]:
E1 = np.linspace(*E1LIM, E1_PTS)
E2 = np.linspace(*E2LIM, E2_PTS)
E1MG, E2MG = np.meshgrid(E1, E2)

E1_BIG = np.linspace(*E1LIM, 100 * E1_PTS)
E2_BIG = np.linspace(*E2LIM, 100 * E2_PTS)
E1MG_BIG, E2MG_BIG = np.meshgrid(E1_BIG, E2_BIG)

def draw_distribution(axes: np.ndarray[mpl_axes.Axes], g: gp.VectorGaussianProcess,
                      which='posterior'):
    if not (which == 'posterior' or which == 'prior'):
        raise ValueError("`which` parameter must be 'posterior' (default) or 'prior'")
    
    x = np.stack((E1MG, E2MG), axis=-1).reshape((-1, 2))

    what = ('truth', 'mean', 'variance', 'error')

    for i in range(axes.shape[0]):
        truth = f(x).reshape((E2_PTS, E1_PTS, g.cdim))[..., i]
        mean = (g.mu_posterior(x) if which == 'posterior' else g.mu(x)).reshape((E2_PTS, E1_PTS, g.cdim))[..., i]
        var = np.diag((g.k_posterior(x) if which == 'posterior' else g.k(x))).reshape((E2_PTS, E1_PTS, g.cdim))[..., i]
        error = np.abs(truth - mean)

        for w, yy, j in zip(what, (truth, mean, var, error), range(axes.shape[1])):
            ax = axes[i, j]

            m = ax.pcolormesh(E1MG, E2MG, yy, **MESH_KWARGS[w])
            plt.colorbar(m, ax=ax)
            ax.set_aspect('equal')
            ax.set_xlim(*E1LIM)
            ax.set_ylim(*E2LIM)
            ax.set_title(w.title())


## True Function and Noisy Observations

Define the true function and a function that yields noisy observations of that true function.

In particular, for this example, let the true function $f: \mathbb{R}^2 \rightarrow \mathbb{R}^2$.  Then noisy observations $\hat{f}(x) = f(x) + \epsilon$ of the true function.  As defined below, $\epsilon \sim \text{N}^2(0, \text{diag}(\sigma^2_1, \sigma^2_2))$ where $\sigma^2_i \sim \text{LogNormal}(-1, 0.8)$.  In general, however, GP methods are able to handle noise that is itself a function of the spatial coordinate, or even more exotic scenarios.


In [ ]:
def f(x):
    a = 2
    return np.stack((np.sin(a * 2 * np.pi * x[..., 0]**2) * x[..., 1],
                     np.cos(a * 2 * np.pi * x[..., 1]**2) * x[..., 0]), axis=-1)

def f_random(x: np.ndarray | float):
    fx = f(x)
    s = sigma(fx)
    return fx + (s.flatten() * np.random.normal(0, 1, s.size)).reshape(s.shape), s

def sigma(fx: np.ndarray):
    # (mu, sigma) = (-1, 0.8) chosen simply for looking nice
    return np.random.lognormal(-1, 0.8, fx.shape)


We plot the true function below:

In [ ]:
coords = (E1MG_BIG, E2MG_BIG)

fig, axes = plt.subplots(2, 2, figsize=(FIGSIZE[0] * 2, FIGSIZE[1] * 2))

for i, tt in zip((0, 1), ('$f_1$', '$f_2$')):
    for j, ff in zip((0, 1), (f, lambda x: f_random(x)[0])):
        ax = axes[i, j]
        m = ax.pcolormesh(*coords,
                          ff(np.stack(coords, axis=-1))[..., i],
                          **MESH_KWARGS['truth'])
        plt.colorbar(m, ax=ax)
        ax.set_aspect('equal')
        ax.set_xlim(*E1LIM)
        ax.set_ylim(*E2LIM)
        ax.set_title(f'{tt}')

plt.tight_layout()
plt.show()


## Defining GP and Prior Mean and Covariance
Define the prior Gaussian process and create the ScalarGaussianProcess object:

In [ ]:
mu = gp.wrapper.VectorFunction(dim=2, cdim=2, f=lambda x: np.zeros((len(x), 2)))
k = gp.kernel.Kronecker(dim=2, cdim=2, C=np.eye(2),  # identity such that this vector GP is composed of independent scalar GPs
                        k=gp.kernel.SquareExponential(dim=2))
g = gp.VectorGaussianProcess(mu, k)


We can plot a realizations of the prior:

In [ ]:
X = np.stack((E1MG, E2MG), axis=-1).reshape((-1, 2))
y = g.sample_prior(X).reshape((E2_PTS, E1_PTS, -1))


fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)

m = ax.pcolormesh(E1MG, E2MG, y[..., 0], **MESH_KWARGS['mean'])
plt.colorbar(m, ax=ax)
ax.set_aspect('equal')
ax.set_xlim(*E1LIM)
ax.set_ylim(*E2LIM)

plt.tight_layout()
plt.show()


Plot statistics of many realizations of the prior:

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(FIGSIZE[0] * 4, FIGSIZE[1] * 2))
draw_distribution(axes, g, which='prior')
plt.tight_layout()
plt.show()


## Conditioned GP and Posterior Mean and Covariance
Now we condition the GP on realizations of the random function $\hat{f}$ (i.e., `f_random` above):

In [ ]:
N_STRT = 10

x_obs = np.stack((E1LIM[0] + np.ptp(E1LIM) * np.random.rand(N_STRT),
                  E2LIM[0] + np.ptp(E2LIM) * np.random.rand(N_STRT)), axis=-1)
y_obs, s_obs = f_random(x_obs)
g.condition(x_obs, y_obs, s_obs)


Now we plot the posterior:


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(FIGSIZE[0] * 4, FIGSIZE[1] * 2))
draw_distribution(axes, g)
for i in range(2):
    for j in range(0, 4, 2):
        ax = axes[i, j]
        ax.scatter(*(x_obs.T), marker='D', c=M.to_rgba(s_obs[..., i]), s=10)
        plt.colorbar(M, ax=ax, extend='max')
        ax.set_aspect('equal')
plt.tight_layout()
plt.show()


We now iteratively update our posterior with observations at the coordinate associated with the highest variance.

In the figures below, the green vertical bar indicates the *coordinate* at which to choose the next sample.  In the figure that follows, the sample chosen at this location is shown with its associated (sampled) variance.

In [ ]:
N_RUN = 10
PLOT_EVERY = 1

for i in range(N_RUN):
    x = X[np.argsort(np.sum(np.diag(g.k_posterior(X)).reshape((len(X), g.cdim))**2, axis=-1))[-1:]]
    y, s = f_random(x)

    x_obs = np.concatenate((x_obs, x), axis=0)
    y_obs = np.concatenate((y_obs, y), axis=0)    
    s_obs = np.concatenate((s_obs, s), axis=0)
    g.condition(x_obs, y_obs, s_obs)

    if (i + 1) % PLOT_EVERY == 0:
        fig, axes = plt.subplots(2, 4, figsize=(FIGSIZE[0] * 4, FIGSIZE[1] * 2))
        draw_distribution(axes, g)
        for i in range(2):
            for j in range(0, 4, 2):
                ax = axes[i, j]
                ax.scatter(*(x_obs[:-1].T), marker='D', c=M.to_rgba(s_obs[:-1, i]), s=10)
                ax.scatter(*(x_obs[-1].T), marker='o', color=M.to_rgba(s_obs[-1, i]), s=200)
                plt.colorbar(M, ax=ax, extend='max')
                ax.set_aspect('equal')
        plt.tight_layout()
        plt.show()

        print('\n' * 3, end='')
